In [ ]:
"""
=============================================================
  AI Insurance Premium Prediction System
  Using Machine Learning (Python)
=============================================================
  Models Used:
    - Linear Regression
    - Ridge Regression
    - Random Forest Regressor
    - Gradient Boosting Regressor
  Features:
    - Synthetic dataset generation
    - Data preprocessing & EDA
    - Feature engineering
    - Model training & comparison
    - Evaluation metrics
    - Prediction interface
    - Visualizations
=============================================================
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import os

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
np.random.seed(42)

# ─────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────
OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]
sns.set_palette(PALETTE)


# ─────────────────────────────────────────────
# 1. SYNTHETIC DATASET GENERATION
# ─────────────────────────────────────────────
def generate_dataset(n=2000):
    """Generate a realistic insurance dataset."""
    age         = np.random.randint(18, 75, n)
    bmi         = np.round(np.random.normal(28, 6, n).clip(15, 55), 1)
    children    = np.random.choice([0, 1, 2, 3, 4, 5], n, p=[0.30, 0.25, 0.22, 0.13, 0.07, 0.03])
    smoker      = np.random.choice(["yes", "no"], n, p=[0.20, 0.80])
    sex         = np.random.choice(["male", "female"], n)
    region      = np.random.choice(["northeast", "northwest", "southeast", "southwest"], n)
    chronic     = np.random.choice([0, 1], n, p=[0.70, 0.30])   # chronic disease flag
    exercise    = np.random.choice(["none", "light", "moderate", "heavy"], n, p=[0.25, 0.30, 0.30, 0.15])
    occupation  = np.random.choice(["white_collar", "blue_collar", "self_employed", "unemployed"], n)

    # Premium formula (realistic actuarial logic)
    base = 3000
    premium = (
        base
        + age * 120
        + (bmi - 25) * 80
        + children * 400
        + (smoker == "yes") * 15000
        + (sex == "male") * 200
        + chronic * 4000
        + (exercise == "none") * 800
        + (exercise == "light") * 300
        + (exercise == "heavy") * -500
        + (occupation == "blue_collar") * 1200
        + (occupation == "self_employed") * 600
        + (occupation == "unemployed") * 900
        + (region == "northeast") * 500
        + (region == "southeast") * -200
        + np.random.normal(0, 1500, n)   # noise
    ).clip(1000, 60000)

    df = pd.DataFrame({
        "age": age, "sex": sex, "bmi": bmi, "children": children,
        "smoker": smoker, "region": region, "chronic_disease": chronic,
        "exercise_level": exercise, "occupation": occupation,
        "premium": np.round(premium, 2)
    })
    return df


# ─────────────────────────────────────────────
# 2. EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────
def eda_plots(df):
    fig = plt.figure(figsize=(20, 16))
    fig.suptitle("Insurance Premium — Exploratory Data Analysis", fontsize=18, fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

    # Premium distribution
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.hist(df["premium"], bins=40, color=PALETTE[0], edgecolor="white", alpha=0.85)
    ax1.set_title("Premium Distribution")
    ax1.set_xlabel("Premium ($)")
    ax1.set_ylabel("Count")

    # Age vs Premium
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(df["age"], df["premium"], alpha=0.3, color=PALETTE[1], s=15)
    ax2.set_title("Age vs Premium")
    ax2.set_xlabel("Age"); ax2.set_ylabel("Premium ($)")

    # BMI vs Premium
    ax3 = fig.add_subplot(gs[0, 2])
    ax3.scatter(df["bmi"], df["premium"], alpha=0.3, color=PALETTE[2], s=15)
    ax3.set_title("BMI vs Premium")
    ax3.set_xlabel("BMI"); ax3.set_ylabel("Premium ($)")

    # Smoker vs Premium
    ax4 = fig.add_subplot(gs[1, 0])
    df.groupby("smoker")["premium"].mean().plot(kind="bar", ax=ax4, color=[PALETTE[3], PALETTE[0]])
    ax4.set_title("Avg Premium by Smoker Status")
    ax4.set_xlabel("Smoker"); ax4.set_ylabel("Avg Premium ($)")
    ax4.tick_params(axis='x', rotation=0)

    # Region vs Premium
    ax5 = fig.add_subplot(gs[1, 1])
    df.groupby("region")["premium"].mean().sort_values(ascending=False).plot(
        kind="bar", ax=ax5, color=PALETTE)
    ax5.set_title("Avg Premium by Region")
    ax5.set_xlabel("Region"); ax5.set_ylabel("Avg Premium ($)")
    ax5.tick_params(axis='x', rotation=20)

    # Exercise vs Premium
    ax6 = fig.add_subplot(gs[1, 2])
    order = ["none", "light", "moderate", "heavy"]
    vals  = [df[df["exercise_level"] == e]["premium"].mean() for e in order]
    ax6.bar(order, vals, color=PALETTE[:4])
    ax6.set_title("Avg Premium by Exercise Level")
    ax6.set_xlabel("Exercise Level"); ax6.set_ylabel("Avg Premium ($)")

    # Children vs Premium
    ax7 = fig.add_subplot(gs[2, 0])
    df.groupby("children")["premium"].mean().plot(kind="bar", ax=ax7, color=PALETTE[0])
    ax7.set_title("Avg Premium by No. of Children")
    ax7.set_xlabel("Children"); ax7.set_ylabel("Avg Premium ($)")

    # Correlation heatmap
    ax8 = fig.add_subplot(gs[2, 1:])
    num_df = df.select_dtypes(include=np.number)
    corr = num_df.corr()
    sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax8, linewidths=0.5)
    ax8.set_title("Correlation Heatmap")

    path = f"{OUTPUT_DIR}/01_eda.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] {path}")


# ─────────────────────────────────────────────
# 3. PREPROCESSING
# ─────────────────────────────────────────────
def preprocess(df):
    df = df.copy()

    # Encode categoricals
    le_smoker   = LabelEncoder().fit(["no", "yes"])
    le_sex      = LabelEncoder().fit(["female", "male"])
    le_region   = LabelEncoder().fit(df["region"].unique())
    le_exercise = LabelEncoder().fit(df["exercise_level"].unique())
    le_occ      = LabelEncoder().fit(df["occupation"].unique())

    df["smoker_enc"]   = le_smoker.transform(df["smoker"])
    df["sex_enc"]      = le_sex.transform(df["sex"])
    df["region_enc"]   = le_region.transform(df["region"])
    df["exercise_enc"] = le_exercise.transform(df["exercise_level"])
    df["occ_enc"]      = le_occ.transform(df["occupation"])

    # Feature engineering
    df["age_bmi"]      = df["age"] * df["bmi"]
    df["age_smoker"]   = df["age"] * df["smoker_enc"]
    df["bmi_smoker"]   = df["bmi"] * df["smoker_enc"]
    df["bmi_category"] = pd.cut(df["bmi"],
                                bins=[0, 18.5, 25, 30, 35, 100],
                                labels=[0, 1, 2, 3, 4]).astype(int)

    feature_cols = [
        "age", "bmi", "children", "smoker_enc", "sex_enc",
        "region_enc", "chronic_disease", "exercise_enc", "occ_enc",
        "age_bmi", "age_smoker", "bmi_smoker", "bmi_category"
    ]

    X = df[feature_cols]
    y = df["premium"]
    return X, y, feature_cols


# ─────────────────────────────────────────────
# 4. MODEL TRAINING & EVALUATION
# ─────────────────────────────────────────────
def train_models(X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    models = {
        "Linear Regression":       LinearRegression(),
        "Ridge Regression":        Ridge(alpha=10),
        "Random Forest":           RandomForestRegressor(n_estimators=200, max_depth=10,
                                                          random_state=42, n_jobs=-1),
        "Gradient Boosting":       GradientBoostingRegressor(n_estimators=200, learning_rate=0.08,
                                                              max_depth=5, random_state=42),
    }

    results = {}
    trained = {}

    print("\n  {'Model':<25} {'MAE':>10} {'RMSE':>10} {'R²':>8} {'CV R²':>10}")
    print("  " + "-" * 65)

    for name, model in models.items():
        # Use scaled data for linear models, raw for tree-based
        use_scaled = "Regression" in name
        Xtr = X_train_sc if use_scaled else X_train.values
        Xte = X_test_sc  if use_scaled else X_test.values

        model.fit(Xtr, y_train)
        preds = model.predict(Xte)

        mae  = mean_absolute_error(y_test, preds)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        r2   = r2_score(y_test, preds)
        cv   = cross_val_score(model, Xtr, y_train, cv=5, scoring="r2").mean()

        results[name] = {"MAE": mae, "RMSE": rmse, "R2": r2, "CV_R2": cv, "preds": preds}
        trained[name] = (model, scaler if use_scaled else None)

        print(f"  {name:<25} {mae:>10,.0f} {rmse:>10,.0f} {r2:>8.4f} {cv:>10.4f}")

    return results, trained


# ─────────────────────────────────────────────
# 5. VISUALIZATIONS — MODEL COMPARISON
# ─────────────────────────────────────────────
def plot_model_comparison(results, y_test):
    names  = list(results.keys())
    maes   = [results[n]["MAE"]  for n in names]
    rmses  = [results[n]["RMSE"] for n in names]
    r2s    = [results[n]["R2"]   for n in names]
    cv_r2s = [results[n]["CV_R2"] for n in names]

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Model Comparison — Insurance Premium Prediction", fontsize=16, fontweight="bold")

    # MAE
    axes[0, 0].barh(names, maes, color=PALETTE)
    axes[0, 0].set_title("Mean Absolute Error (lower = better)")
    axes[0, 0].set_xlabel("MAE ($)")

    # RMSE
    axes[0, 1].barh(names, rmses, color=PALETTE)
    axes[0, 1].set_title("Root Mean Squared Error (lower = better)")
    axes[0, 1].set_xlabel("RMSE ($)")

    # R²
    axes[1, 0].barh(names, r2s, color=PALETTE)
    axes[1, 0].set_title("R² Score (higher = better)")
    axes[1, 0].set_xlabel("R²")
    axes[1, 0].axvline(1.0, color="black", linestyle="--", alpha=0.4)

    # CV R²
    axes[1, 1].barh(names, cv_r2s, color=PALETTE)
    axes[1, 1].set_title("Cross-Validation R² (5-fold)")
    axes[1, 1].set_xlabel("CV R²")

    plt.tight_layout()
    path = f"{OUTPUT_DIR}/02_model_comparison.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] {path}")

    # Actual vs Predicted (best model = Gradient Boosting)
    best_name = max(results, key=lambda n: results[n]["R2"])
    preds     = results[best_name]["preds"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"Best Model: {best_name}", fontsize=14, fontweight="bold")

    axes[0].scatter(y_test, preds, alpha=0.35, color=PALETTE[0], s=12)
    mn, mx = y_test.min(), y_test.max()
    axes[0].plot([mn, mx], [mn, mx], "r--", lw=1.5)
    axes[0].set_title("Actual vs Predicted Premium")
    axes[0].set_xlabel("Actual ($)"); axes[0].set_ylabel("Predicted ($)")

    residuals = y_test.values - preds
    axes[1].hist(residuals, bins=50, color=PALETTE[1], edgecolor="white", alpha=0.85)
    axes[1].axvline(0, color="red", linestyle="--")
    axes[1].set_title("Residuals Distribution")
    axes[1].set_xlabel("Residual ($)"); axes[1].set_ylabel("Count")

    plt.tight_layout()
    path = f"{OUTPUT_DIR}/03_actual_vs_predicted.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] {path}")


# ─────────────────────────────────────────────
# 6. FEATURE IMPORTANCE
# ─────────────────────────────────────────────
def plot_feature_importance(trained, feature_cols):
    model, _ = trained["Gradient Boosting"]
    importances = model.feature_importances_
    fi = pd.Series(importances, index=feature_cols).sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(fi)))
    fi.plot(kind="barh", ax=ax, color=colors)
    ax.set_title("Feature Importance — Gradient Boosting Model", fontsize=14, fontweight="bold")
    ax.set_xlabel("Importance Score")
    for bar, val in zip(ax.patches, fi.values):
        ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
                f"{val:.4f}", va="center", fontsize=9)
    plt.tight_layout()
    path = f"{OUTPUT_DIR}/04_feature_importance.png"
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  [Saved] {path}")


# ─────────────────────────────────────────────
# 7. PREMIUM PREDICTOR INTERFACE
# ─────────────────────────────────────────────
class InsurancePremiumPredictor:
    """Simple prediction interface using the best trained model."""

    EXERCISE_MAP  = {"none": 0, "light": 1, "moderate": 2, "heavy": 3}
    REGION_MAP    = {"northeast": 0, "northwest": 1, "southeast": 2, "southwest": 3}
    OCC_MAP       = {"blue_collar": 0, "self_employed": 1, "unemployed": 2, "white_collar": 3}
    BMI_CATEGORY  = lambda self, bmi: (0 if bmi < 18.5 else 1 if bmi < 25 else
                                       2 if bmi < 30 else 3 if bmi < 35 else 4)

    def __init__(self, model):
        self.model = model  # GradientBoostingRegressor (no scaler needed)

    def predict(self, age, bmi, children, smoker, sex, region,
                chronic_disease, exercise_level, occupation):
        s  = 1 if smoker == "yes" else 0
        sx = 1 if sex == "male" else 0
        r  = self.REGION_MAP.get(region, 0)
        ex = self.EXERCISE_MAP.get(exercise_level, 0)
        oc = self.OCC_MAP.get(occupation, 0)
        bc = self.BMI_CATEGORY(bmi)

        features = np.array([[
            age, bmi, children, s, sx, r,
            chronic_disease, ex, oc,
            age * bmi, age * s, bmi * s, bc
        ]])
        return self.model.predict(features)[0]


# ─────────────────────────────────────────────
# 8. SAMPLE PREDICTIONS TABLE
# ─────────────────────────────────────────────
def show_sample_predictions(predictor):
    samples = [
        (25, 22.0, 0, "no",  "female", "northeast", 0, "heavy",    "white_collar"),
        (40, 30.5, 2, "no",  "male",   "southeast", 1, "light",    "blue_collar"),
        (55, 35.0, 3, "yes", "male",   "southwest", 1, "none",     "self_employed"),
        (30, 27.8, 1, "no",  "female", "northwest", 0, "moderate", "white_collar"),
        (65, 28.0, 0, "yes", "male",   "northeast", 1, "none",     "unemployed"),
        (22, 19.5, 0, "no",  "female", "southeast", 0, "heavy",    "white_collar"),
    ]
    print("\n  Sample Predictions")
    print("  " + "─" * 80)
    print(f"  {'Age':>4} {'BMI':>6} {'Kids':>5} {'Smoke':>6} {'Sex':>7} {'Region':>11} "
          f"{'Chronic':>8} {'Exercise':>10} {'Predicted':>12}")
    print("  " + "─" * 80)
    for s in samples:
        pred = predictor.predict(*s)
        print(f"  {s[0]:>4} {s[1]:>6.1f} {s[2]:>5} {s[3]:>6} {s[4]:>7} {s[5]:>11} "
              f"{s[6]:>8} {s[7]:>10}  ${pred:>10,.2f}")


# ─────────────────────────────────────────────
# 9. SUMMARY STATS TABLE (TEXT)
# ─────────────────────────────────────────────
def print_summary(df):
    print("\n  Dataset Summary")
    print("  " + "─" * 45)
    print(f"  Total Records   : {len(df):,}")
    print(f"  Premium Range   : ${df['premium'].min():,.0f} – ${df['premium'].max():,.0f}")
    print(f"  Mean Premium    : ${df['premium'].mean():,.2f}")
    print(f"  Median Premium  : ${df['premium'].median():,.2f}")
    print(f"  Smokers         : {(df['smoker']=='yes').mean()*100:.1f}%")
    print(f"  Chronic Disease : {df['chronic_disease'].mean()*100:.1f}%")
    print(f"  Avg Age         : {df['age'].mean():.1f}")
    print(f"  Avg BMI         : {df['bmi'].mean():.1f}")


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
def main():
    print("\n" + "=" * 65)
    print("   AI INSURANCE PREMIUM PREDICTION SYSTEM")
    print("=" * 65)

    # 1. Generate dataset
    print("\n[1] Generating synthetic insurance dataset...")
    df = generate_dataset(n=2000)
    print(f"    Dataset shape: {df.shape}")
    print_summary(df)

    # 2. EDA
    print("\n[2] Generating EDA visualizations...")
    eda_plots(df)

    # 3. Preprocessing
    print("\n[3] Preprocessing & feature engineering...")
    X, y, feature_cols = preprocess(df)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"    Train: {X_train.shape}  |  Test: {X_test.shape}")

    # 4. Train models
    print("\n[4] Training & evaluating models...")
    results, trained = train_models(X_train, X_test, y_train, y_test)

    # 5. Plot comparisons
    print("\n[5] Generating model comparison plots...")
    plot_model_comparison(results, y_test)

    # 6. Feature importance
    print("\n[6] Generating feature importance plot...")
    plot_feature_importance(trained, feature_cols)

    # 7. Predictor
    print("\n[7] Running sample predictions with best model (Gradient Boosting)...")
    gb_model, _ = trained["Gradient Boosting"]
    predictor = InsurancePremiumPredictor(gb_model)
    show_sample_predictions(predictor)

    best = max(results, key=lambda n: results[n]["R2"])
    print(f"\n  ✓ Best Model   : {best}")
    print(f"  ✓ R² Score     : {results[best]['R2']:.4f}")
    print(f"  ✓ MAE          : ${results[best]['MAE']:,.2f}")
    print(f"  ✓ RMSE         : ${results[best]['RMSE']:,.2f}")

    print("\n" + "=" * 65)
    print("   All outputs saved to /mnt/user-data/outputs/")
    print("=" * 65 + "\n")


if __name__ == "__main__":
    main()


   AI INSURANCE PREMIUM PREDICTION SYSTEM

[1] Generating synthetic insurance dataset...
    Dataset shape: (2000, 10)

  Dataset Summary
  ─────────────────────────────────────────────
  Total Records   : 2,000
  Premium Range   : $3,730 – $34,908
  Mean Premium    : $14,900.11
  Median Premium  : $12,850.17
  Smokers         : 20.8%
  Chronic Disease : 31.2%
  Avg Age         : 46.4
  Avg BMI         : 28.3

[2] Generating EDA visualizations...
  [Saved] /mnt/user-data/outputs/01_eda.png

[3] Preprocessing & feature engineering...
    Train: (1600, 13)  |  Test: (400, 13)

[4] Training & evaluating models...

  {'Model':<25} {'MAE':>10} {'RMSE':>10} {'R²':>8} {'CV R²':>10}
  -----------------------------------------------------------------
  Linear Regression              1,132      1,413   0.9589     0.9444
  Ridge Regression               1,144      1,420   0.9585     0.9436
